# Stress Prediction v6 — Feature Engineering + ExtraTrees Seed Ensemble

이 노트북은 실제 `train.csv` / `test.csv` 구조에 맞춘 최종 실험용 버전입니다.

## 설계 원칙

1. 기존에 가장 강했던 ExtraTrees 구조를 기준선으로 정확히 유지
2. 파생변수를 무조건 전부 넣지 않고 **그룹별 OOF 검증**
3. 기준선보다 좋아진 feature set만 최종 단계에 사용
4. 서로 다른 random seed의 ExtraTrees 예측을 합쳐 분산 감소
5. 기존 Pair predictor와 nearest-neighbor override 유지
6. tree quantile, pair quantile, weight, rounding, override threshold 자동 탐색
7. 성격이 다른 제출 후보 최대 5개 생성

> 실행 시간이 부담되면 `SCREEN_N_ESTIMATORS`, `FINAL_N_ESTIMATORS`, `SEEDS`를 줄여도 됩니다.


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"

RANDOM_SEED = 42
N_SPLITS = 5

# 빠른 feature group 선별용
SCREEN_N_ESTIMATORS = 600

# 최종 seed ensemble용
FINAL_N_ESTIMATORS = 1000
SEEDS = [42, 52, 62]

# 기존 구조 주변 탐색 범위
TREE_QUANTILES = [0.48, 0.50, 0.52, 0.54, 0.56]
PAIR_QUANTILES = [0.46, 0.48, 0.50]
PAIR_WEIGHTS = [0.10, 0.15, 0.20, 0.25]
OVERRIDE_THRESHOLDS = [None, 0.12, 0.15, 0.18, 0.20, 0.22]
ROUND_OPTIONS = [2, None]

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

# 로컬 환경 경로
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 첨부 파일 fallback
TRAIN_FALLBACK = Path("/mnt/data/train(2).csv")
TEST_FALLBACK = Path("/mnt/data/test(2).csv")

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Screen trees:", SCREEN_N_ESTIMATORS)
print("Final trees per seed:", FINAL_N_ESTIMATORS)
print("Seeds:", SEEDS)


DATA_DIR: /Users/bitsaem/Desktop/stress_project/open
OUTPUT_DIR: /Users/bitsaem/Desktop/stress_project/8_6
Screen trees: 600
Final trees per seed: 1000
Seeds: [42, 52, 62]


## 1. 데이터 로드 및 구조 확인


In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

if not train_path.exists() and TRAIN_FALLBACK.exists():
    train_path = TRAIN_FALLBACK

if not test_path.exists() and TEST_FALLBACK.exists():
    test_path = TEST_FALLBACK

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

expected_train_columns = [
    "ID", "gender", "age", "height", "weight", "cholesterol",
    "systolic_blood_pressure", "diastolic_blood_pressure",
    "glucose", "bone_density", "activity", "smoke_status",
    "medical_history", "family_medical_history", "sleep_pattern",
    "edu_level", "mean_working", "stress_score",
]

missing = set(expected_train_columns) - set(train_raw.columns)
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing)}")

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
display(train_raw.head())
display(train_raw.dtypes.to_frame("dtype"))

y = train_raw[TARGET].reset_index(drop=True)
X_raw = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)
X_test_raw = (
    test_raw
    .drop(columns=[ID_COLUMN])
    .reset_index(drop=True)
)

numeric_raw_columns = X_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_raw_columns = X_raw.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Numeric:", numeric_raw_columns)
print("Categorical:", categorical_raw_columns)
print("\nTarget summary")
display(y.describe().to_frame().T)


## 2. Train/Test 분포 차이 점검


In [3]:
distribution_rows = []

for col in numeric_raw_columns:
    train_col = X_raw[col]
    test_col = X_test_raw[col]

    pooled_std = pd.concat([train_col, test_col]).std()
    mean_shift = (
        abs(train_col.mean() - test_col.mean()) / (pooled_std + 1e-9)
    )

    distribution_rows.append({
        "feature": col,
        "train_mean": train_col.mean(),
        "test_mean": test_col.mean(),
        "standardized_mean_shift": mean_shift,
        "train_missing": train_col.isna().mean(),
        "test_missing": test_col.isna().mean(),
    })

distribution_df = (
    pd.DataFrame(distribution_rows)
    .sort_values("standardized_mean_shift", ascending=False)
)

display(distribution_df)

category_rows = []
for col in categorical_raw_columns:
    train_levels = set(X_raw[col].dropna().astype(str))
    test_levels = set(X_test_raw[col].dropna().astype(str))

    category_rows.append({
        "feature": col,
        "train_unique": len(train_levels),
        "test_unique": len(test_levels),
        "only_train": sorted(train_levels - test_levels),
        "only_test": sorted(test_levels - train_levels),
    })

display(pd.DataFrame(category_rows))


,feature,train_mean,test_mean,standardized_mean_shift,train_missing,test_missing
8,mean_working,8.716972,8.844378,0.078363,0.344,0.336
6,glucose,126.467720,126.949247,0.026424,0.000,0.000
0,age,53.068000,53.397667,0.016098,0.000,0.000
3,cholesterol,233.910523,234.296067,0.015725,0.000,0.000
4,systolic_blood_pressure,146.144333,145.912667,0.014502,0.000,0.000
2,weight,72.486680,72.667163,0.013685,0.000,0.000
5,diastolic_blood_pressure,95.504000,95.637000,0.013386,0.000,0.000
1,height,168.585687,168.651740,0.007108,0.000,0.000
7,bone_density,0.933367,0.933303,0.000143,0.000,0.000


,feature,train_unique,test_unique,only_train,only_test
0,gender,2,2,[],[]
1,activity,3,3,[],[]
2,smoke_status,3,3,[],[]
3,medical_history,3,3,[],[]
4,family_medical_history,3,3,[],[]
5,sleep_pattern,3,3,[],[]
6,edu_level,3,3,[],[]


## 3. Feature engineering 함수


In [4]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series,
) -> pd.Series:
    return numerator / (denominator.abs() + 1e-6)


def add_base_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = safe_divide(
        result["weight"],
        height_m.pow(2),
    )
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = safe_divide(sbp, dbp)
    result["cholesterol_glucose_ratio"] = safe_divide(
        result["cholesterol"],
        result["glucose"],
    )
    result["missing_count"] = frame.isna().sum(axis=1)

    return result


def add_body_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    result["bmi_squared"] = result["bmi"].pow(2)
    result["bmi_cubed"] = result["bmi"].pow(3)
    result["weight_squared"] = result["weight"].pow(2)
    result["height_squared"] = result["height"].pow(2)
    result["weight_height_ratio"] = safe_divide(
        result["weight"],
        result["height"],
    )
    result["age_bmi"] = result["age"] * result["bmi"]
    result["age_weight"] = result["age"] * result["weight"]
    result["bone_bmi"] = result["bone_density"] * result["bmi"]
    result["bone_weight_ratio"] = safe_divide(
        result["bone_density"],
        result["weight"],
    )
    result["bone_age"] = result["bone_density"] * result["age"]

    return result


def add_blood_pressure_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    pulse = result["pulse_pressure"]
    map_value = result["mean_arterial_pressure"]

    result["sbp_squared"] = sbp.pow(2)
    result["dbp_squared"] = dbp.pow(2)
    result["bp_product"] = sbp * dbp
    result["bp_sum"] = sbp + dbp
    result["pulse_pressure_squared"] = pulse.pow(2)
    result["map_squared"] = map_value.pow(2)
    result["age_sbp"] = result["age"] * sbp
    result["age_dbp"] = result["age"] * dbp
    result["bmi_sbp"] = result["bmi"] * sbp
    result["bmi_dbp"] = result["bmi"] * dbp

    return result


def add_metabolic_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    chol = result["cholesterol"]
    glucose = result["glucose"]
    bmi = result["bmi"]

    result["cholesterol_squared"] = chol.pow(2)
    result["glucose_squared"] = glucose.pow(2)
    result["cholesterol_glucose_product"] = chol * glucose
    result["cholesterol_bmi"] = chol * bmi
    result["glucose_bmi"] = glucose * bmi
    result["cholesterol_bmi_ratio"] = safe_divide(chol, bmi)
    result["glucose_bmi_ratio"] = safe_divide(glucose, bmi)
    result["age_cholesterol"] = result["age"] * chol
    result["age_glucose"] = result["age"] * glucose
    result["bone_cholesterol"] = result["bone_density"] * chol
    result["bone_glucose"] = result["bone_density"] * glucose

    return result


def add_working_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    working = result["mean_working"]

    result["working_squared"] = working.pow(2)
    result["working_cubed"] = working.pow(3)
    result["working_age"] = working * result["age"]
    result["working_bmi"] = working * result["bmi"]
    result["working_weight"] = working * result["weight"]
    result["working_bone"] = working * result["bone_density"]
    result["working_glucose"] = working * result["glucose"]
    result["working_cholesterol"] = working * result["cholesterol"]
    result["working_pulse_pressure"] = working * result["pulse_pressure"]
    result["working_map"] = working * result["mean_arterial_pressure"]

    return result


def add_numeric_log_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    log_columns = [
        "age", "height", "weight", "cholesterol", "glucose",
        "mean_working", "bmi", "pulse_pressure",
        "mean_arterial_pressure",
    ]

    for col in log_columns:
        minimum = result[col].min(skipna=True)
        shifted = result[col] - min(minimum, 0) + 1e-6
        result[f"{col}_log1p"] = np.log1p(shifted)

    return result


CATEGORICAL_CROSSES = [
    ("gender", "activity"),
    ("gender", "smoke_status"),
    ("gender", "sleep_pattern"),
    ("activity", "sleep_pattern"),
    ("activity", "edu_level"),
    ("activity", "smoke_status"),
    ("smoke_status", "medical_history"),
    ("medical_history", "family_medical_history"),
    ("sleep_pattern", "edu_level"),
    ("gender", "medical_history"),
]


def add_categorical_crosses(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    for first, second in CATEGORICAL_CROSSES:
        first_value = result[first].fillna("__MISSING__").astype(str)
        second_value = result[second].fillna("__MISSING__").astype(str)
        result[f"cross__{first}__{second}"] = (
            first_value + "__" + second_value
        )

    return result


def build_feature_set(
    frame: pd.DataFrame,
    feature_set: str,
) -> pd.DataFrame:
    result = add_base_features(frame)

    if feature_set == "baseline":
        return result

    if feature_set in {
        "body",
        "numeric_core",
        "numeric_all",
        "all_with_crosses",
    }:
        result = add_body_features(result)

    if feature_set in {
        "blood_pressure",
        "numeric_core",
        "numeric_all",
        "all_with_crosses",
    }:
        result = add_blood_pressure_features(result)

    if feature_set in {
        "metabolic",
        "numeric_core",
        "numeric_all",
        "all_with_crosses",
    }:
        result = add_metabolic_features(result)

    if feature_set in {
        "working",
        "numeric_core",
        "numeric_all",
        "all_with_crosses",
    }:
        result = add_working_features(result)

    if feature_set in {"numeric_all", "all_with_crosses"}:
        result = add_numeric_log_features(result)

    if feature_set in {"categorical_crosses", "all_with_crosses"}:
        result = add_categorical_crosses(result)

    return result


FEATURE_SETS = [
    "baseline",
    "body",
    "blood_pressure",
    "metabolic",
    "working",
    "categorical_crosses",
    "numeric_core",
    "numeric_all",
    "all_with_crosses",
]

for feature_set in FEATURE_SETS:
    example = build_feature_set(X_raw.head(10), feature_set)
    print(feature_set, "->", example.shape[1], "features")


baseline -> 22 features
body -> 32 features
blood_pressure -> 32 features
metabolic -> 33 features
working -> 32 features
categorical_crosses -> 32 features
numeric_core -> 63 features
numeric_all -> 72 features
all_with_crosses -> 82 features


## 4. 기존 ExtraTrees 구조 및 공통 함수


In [5]:
def compute_iqr_bounds(
    train_frame: pd.DataFrame,
    columns: list[str],
) -> dict[str, tuple[float, float]]:
    bounds = {}

    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr,
        )

    return bounds


def apply_winsorize(
    frame: pd.DataFrame,
    bounds: dict[str, tuple[float, float]],
) -> pd.DataFrame:
    result = frame.copy()

    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)

    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue

        for copy_idx in range(1, n_copies + 1):
            result[f"{feature}__copy{copy_idx}"] = result[feature]

    return result


def make_dense_preprocessor(
    frame: pd.DataFrame,
) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numerical = frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing",
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numerical,
        ),
    ])


def fit_extra_trees_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    n_estimators: int,
    random_seed: int,
    return_importance: bool = False,
):
    # 원본 설정 유지: gender 제거 + feature copy + max_features=1
    train_no_gender = train_features.drop(
        columns=["gender"],
        errors="ignore",
    )
    val_no_gender = val_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)

    preprocessor = make_dense_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)

    model = ExtraTreesRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=random_seed,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)

    prediction_matrix = np.column_stack([
        tree.predict(val_matrix)
        for tree in model.estimators_
    ])

    if not return_importance:
        return prediction_matrix

    try:
        feature_names = preprocessor.get_feature_names_out()
        importance = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
    except Exception:
        importance = pd.DataFrame()

    return prediction_matrix, importance


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
):
    median = np.nanmedian(train_values)

    train_values = np.nan_to_num(
        train_values,
        nan=median,
    )
    test_values = np.nan_to_num(
        test_values,
        nan=median,
    )

    sorted_train = np.sort(train_values)

    train_rank = (
        np.searchsorted(
            sorted_train,
            train_values,
            side="right",
        )
        / len(train_values)
    )
    test_rank = (
        np.searchsorted(
            sorted_train,
            test_values,
            side="right",
        )
        / len(train_values)
    )

    return (
        train_rank.astype(np.float32),
        test_rank.astype(np.float32),
    )


def pair_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    train_rank = {}
    val_rank = {}

    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = (
            empirical_rank_transform(
                train_features[column].to_numpy(dtype=float),
                val_features[column].to_numpy(dtype=float),
            )
        )

    target_values = target.to_numpy(dtype=float)
    pair_targets = []

    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )

        nearest_idx = np.argmin(distance, axis=1)
        pair_targets.append(target_values[nearest_idx])

    return np.column_stack(pair_targets)


def nearest_neighbor_lookup(
    train_raw_frame: pd.DataFrame,
    val_raw_frame: pd.DataFrame,
    target: pd.Series,
):
    train_numeric = train_raw_frame[numeric_raw_columns].copy()
    val_numeric = val_raw_frame[numeric_raw_columns].copy()

    medians = train_numeric.median()
    train_numeric = train_numeric.fillna(medians)
    val_numeric = val_numeric.fillna(medians)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)

    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(
        val_scaled,
        k=1,
    )

    nearest_target = (
        target.to_numpy(dtype=float)[nearest_idx]
    )

    return nearest_dist, nearest_target


def apply_postprocess(
    prediction: np.ndarray,
    round_decimals: int | None,
    override_threshold: float | None,
    nearest_dist: np.ndarray,
    nearest_target: np.ndarray,
) -> np.ndarray:
    result = np.clip(prediction.copy(), 0.0, 1.0)

    if round_decimals is not None:
        result = np.round(result, round_decimals)

    if override_threshold is not None:
        mask = nearest_dist < override_threshold
        replacement = nearest_target[mask].copy()

        if round_decimals is not None:
            replacement = np.round(
                replacement,
                round_decimals,
            )

        result[mask] = np.clip(
            replacement,
            0.0,
            1.0,
        )

    return result


## 5. Feature set 그룹별 빠른 OOF 검증

- 모든 feature set을 동일한 CV split으로 비교합니다.
- 이 단계는 `SCREEN_N_ESTIMATORS`를 사용하므로 최종 모델보다 빠릅니다.
- 기준선보다 성능이 나쁜 feature set은 최종 단계에서 자동 제외됩니다.


In [6]:
cv_splits = list(
    KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    ).split(X_raw)
)

screen_results = []
screen_oof_predictions = {}

for feature_set in FEATURE_SETS:
    oof_pred = np.zeros(len(y), dtype=float)

    for fold, (train_idx, val_idx) in enumerate(
        cv_splits,
        start=1,
    ):
        train_fold_raw = (
            X_raw.iloc[train_idx]
            .reset_index(drop=True)
        )
        val_fold_raw = (
            X_raw.iloc[val_idx]
            .reset_index(drop=True)
        )
        y_train = (
            y.iloc[train_idx]
            .reset_index(drop=True)
        )

        bounds = compute_iqr_bounds(
            train_fold_raw,
            WINSORIZE_COLUMNS,
        )
        train_fold_w = apply_winsorize(
            train_fold_raw,
            bounds,
        )
        val_fold_w = apply_winsorize(
            val_fold_raw,
            bounds,
        )

        train_features = build_feature_set(
            train_fold_w,
            feature_set,
        )
        val_features = build_feature_set(
            val_fold_w,
            feature_set,
        )

        tree_matrix = fit_extra_trees_matrix(
            train_features,
            val_features,
            y_train,
            n_estimators=SCREEN_N_ESTIMATORS,
            random_seed=RANDOM_SEED,
        )

        pred = np.quantile(
            tree_matrix,
            0.52,
            axis=1,
        )
        pred = np.round(
            np.clip(pred, 0.0, 1.0),
            2,
        )

        oof_pred[val_idx] = pred

        print(
            f"[{feature_set}] Fold {fold}/{N_SPLITS} 완료"
        )

    mae = mean_absolute_error(y, oof_pred)

    screen_results.append({
        "feature_set": feature_set,
        "feature_count": build_feature_set(
            X_raw.head(10),
            feature_set,
        ).shape[1],
        "cv_mae": mae,
    })
    screen_oof_predictions[feature_set] = oof_pred

    print(
        f"{feature_set}: CV MAE={mae:.8f}\n"
    )

screen_df = (
    pd.DataFrame(screen_results)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

baseline_mae = float(
    screen_df.loc[
        screen_df["feature_set"] == "baseline",
        "cv_mae",
    ].iloc[0]
)

screen_df["improvement_vs_baseline"] = (
    baseline_mae - screen_df["cv_mae"]
)

display(screen_df)


[baseline] Fold 1/5 완료
[baseline] Fold 2/5 완료
[baseline] Fold 3/5 완료
[baseline] Fold 4/5 완료
[baseline] Fold 5/5 완료
baseline: CV MAE=0.14725667

[body] Fold 1/5 완료
[body] Fold 2/5 완료
[body] Fold 3/5 완료
[body] Fold 4/5 완료
[body] Fold 5/5 완료
body: CV MAE=0.14755000

[blood_pressure] Fold 1/5 완료
[blood_pressure] Fold 2/5 완료
[blood_pressure] Fold 3/5 완료
[blood_pressure] Fold 4/5 완료
[blood_pressure] Fold 5/5 완료
blood_pressure: CV MAE=0.14795667

[metabolic] Fold 1/5 완료
[metabolic] Fold 2/5 완료
[metabolic] Fold 3/5 완료
[metabolic] Fold 4/5 완료
[metabolic] Fold 5/5 완료
metabolic: CV MAE=0.14817000

[working] Fold 1/5 완료
[working] Fold 2/5 완료
[working] Fold 3/5 완료
[working] Fold 4/5 완료
[working] Fold 5/5 완료
working: CV MAE=0.16199667

[categorical_crosses] Fold 1/5 완료
[categorical_crosses] Fold 2/5 완료
[categorical_crosses] Fold 3/5 완료
[categorical_crosses] Fold 4/5 완료
[categorical_crosses] Fold 5/5 완료
categorical_crosses: CV MAE=0.15384000

[numeric_core] Fold 1/5 완료
[numeric_core] Fold 2/5 완료
[num

,feature_set,feature_count,cv_mae,improvement_vs_baseline
0,baseline,22,0.147257,0.000000
1,body,32,0.147550,-0.000293
2,blood_pressure,32,0.147957,-0.000700
3,metabolic,33,0.148170,-0.000913
4,all_with_crosses,82,0.152187,-0.004930
5,categorical_crosses,32,0.153840,-0.006583
6,numeric_all,72,0.154523,-0.007267
7,numeric_core,63,0.154783,-0.007527
8,working,32,0.161997,-0.014740


## 6. 최종 feature set 자동 선택 및 중요도 확인


In [7]:
best_screen_row = screen_df.iloc[0]
BEST_FEATURE_SET = str(best_screen_row["feature_set"])

# 개선이 너무 작으면 과적합 방지를 위해 baseline 유지
MIN_REQUIRED_IMPROVEMENT = 0.00020

best_improvement = float(
    best_screen_row["improvement_vs_baseline"]
)

if (
    BEST_FEATURE_SET != "baseline"
    and best_improvement < MIN_REQUIRED_IMPROVEMENT
):
    print(
        "최고 feature set의 개선 폭이 작아 "
        "안전하게 baseline을 사용합니다."
    )
    BEST_FEATURE_SET = "baseline"

print("선택된 최종 feature set:", BEST_FEATURE_SET)
print("기준선 MAE:", baseline_mae)
print("최고 개선 폭:", best_improvement)

# 전체 데이터에서 중요도는 해석용으로만 계산
full_bounds = compute_iqr_bounds(
    X_raw,
    WINSORIZE_COLUMNS,
)
full_w = apply_winsorize(
    X_raw,
    full_bounds,
)
full_features = build_feature_set(
    full_w,
    BEST_FEATURE_SET,
)

_, importance_df = fit_extra_trees_matrix(
    full_features,
    full_features.iloc[:10].copy(),
    y,
    n_estimators=SCREEN_N_ESTIMATORS,
    random_seed=RANDOM_SEED,
    return_importance=True,
)

if len(importance_df):
    print("상위 feature importance")
    display(importance_df.head(40))


선택된 최종 feature set: baseline
기준선 MAE: 0.14725666666666665
최고 개선 폭: 0.0
상위 feature importance


,feature,importance
46,numerical__glucose__copy2,0.024161
48,numerical__weight__copy1,0.023848
24,numerical__cholesterol,0.023779
22,numerical__height,0.023727
43,numerical__height__copy1,0.023710
41,numerical__cholesterol__copy1,0.023685
23,numerical__weight,0.023649
37,numerical__bmi__copy1,0.023628
30,numerical__bmi,0.023620
42,numerical__cholesterol__copy2,0.023611


## 7. 최종 Seed Ensemble OOF 캐시 생성

각 fold에서 여러 seed의 ExtraTrees를 학습합니다.  
각 seed의 개별 tree 예측을 합쳐 quantile을 계산하므로 단일 seed보다 안정적일 수 있습니다.


In [8]:
final_fold_cache = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1,
):
    train_fold_raw = (
        X_raw.iloc[train_idx]
        .reset_index(drop=True)
    )
    val_fold_raw = (
        X_raw.iloc[val_idx]
        .reset_index(drop=True)
    )

    y_train = (
        y.iloc[train_idx]
        .reset_index(drop=True)
    )

    bounds = compute_iqr_bounds(
        train_fold_raw,
        WINSORIZE_COLUMNS,
    )
    train_fold_w = apply_winsorize(
        train_fold_raw,
        bounds,
    )
    val_fold_w = apply_winsorize(
        val_fold_raw,
        bounds,
    )

    train_features = build_feature_set(
        train_fold_w,
        BEST_FEATURE_SET,
    )
    val_features = build_feature_set(
        val_fold_w,
        BEST_FEATURE_SET,
    )

    seed_tree_matrices = []

    for seed in SEEDS:
        tree_matrix = fit_extra_trees_matrix(
            train_features,
            val_features,
            y_train,
            n_estimators=FINAL_N_ESTIMATORS,
            random_seed=seed,
        )
        seed_tree_matrices.append(tree_matrix)

        print(
            f"Fold {fold}: ExtraTrees seed {seed} 완료"
        )

    # shape: n_val × (n_trees × n_seeds)
    combined_tree_matrix = np.concatenate(
        seed_tree_matrices,
        axis=1,
    )

    # Pair는 기존 base feature만 사용
    train_base = add_base_features(train_fold_w)
    val_base = add_base_features(val_fold_w)

    pair_matrix = pair_prediction_matrix(
        train_base,
        val_base,
        y_train,
    )

    nearest_dist, nearest_target = nearest_neighbor_lookup(
        train_fold_raw,
        val_fold_raw,
        y_train,
    )

    final_fold_cache.append({
        "val_idx": val_idx,
        "tree_matrix": combined_tree_matrix,
        "pair_matrix": pair_matrix,
        "nearest_dist": nearest_dist,
        "nearest_target": nearest_target,
    })

    print(f"Fold {fold}/{N_SPLITS} 최종 캐시 완료\n")


Fold 1: ExtraTrees seed 42 완료
Fold 1: ExtraTrees seed 52 완료
Fold 1: ExtraTrees seed 62 완료
Fold 1/5 최종 캐시 완료

Fold 2: ExtraTrees seed 42 완료
Fold 2: ExtraTrees seed 52 완료
Fold 2: ExtraTrees seed 62 완료
Fold 2/5 최종 캐시 완료

Fold 3: ExtraTrees seed 42 완료
Fold 3: ExtraTrees seed 52 완료
Fold 3: ExtraTrees seed 62 완료
Fold 3/5 최종 캐시 완료

Fold 4: ExtraTrees seed 42 완료
Fold 4: ExtraTrees seed 52 완료
Fold 4: ExtraTrees seed 62 완료
Fold 4/5 최종 캐시 완료

Fold 5: ExtraTrees seed 42 완료
Fold 5: ExtraTrees seed 52 완료
Fold 5: ExtraTrees seed 62 완료
Fold 5/5 최종 캐시 완료



## 8. Quantile · Pair weight · Override 자동 탐색


In [9]:
search_rows = []

for tree_q in TREE_QUANTILES:
    for pair_q in PAIR_QUANTILES:
        for pair_weight in PAIR_WEIGHTS:
            for round_decimals in ROUND_OPTIONS:
                for threshold in OVERRIDE_THRESHOLDS:
                    oof_pred = np.zeros(
                        len(y),
                        dtype=float,
                    )
                    override_count = 0

                    for cache in final_fold_cache:
                        tree_pred = np.quantile(
                            cache["tree_matrix"],
                            tree_q,
                            axis=1,
                        )
                        pair_pred = np.quantile(
                            cache["pair_matrix"],
                            pair_q,
                            axis=1,
                        )

                        blended = (
                            (1.0 - pair_weight) * tree_pred
                            + pair_weight * pair_pred
                        )

                        final_pred = apply_postprocess(
                            blended,
                            round_decimals,
                            threshold,
                            cache["nearest_dist"],
                            cache["nearest_target"],
                        )

                        oof_pred[cache["val_idx"]] = (
                            final_pred
                        )

                        if threshold is not None:
                            override_count += int(
                                (
                                    cache["nearest_dist"]
                                    < threshold
                                ).sum()
                            )

                    mae = mean_absolute_error(
                        y,
                        oof_pred,
                    )

                    search_rows.append({
                        "cv_mae": mae,
                        "tree_q": tree_q,
                        "pair_q": pair_q,
                        "pair_weight": pair_weight,
                        "round_decimals": round_decimals,
                        "override_threshold": threshold,
                        "override_rows": override_count,
                        "feature_set": BEST_FEATURE_SET,
                        "seed_count": len(SEEDS),
                    })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

print("상위 30개 조합")
display(search_df.head(30))

print(
    "최적 최종 OOF MAE:",
    float(search_df.iloc[0]["cv_mae"]),
)
print(
    "기존 screen baseline MAE:",
    baseline_mae,
)


상위 30개 조합


,cv_mae,tree_q,pair_q,pair_weight,round_decimals,override_threshold,override_rows,feature_set,seed_count
0,0.147407,0.54,0.46,0.25,2.0,0.20,572,baseline,3
1,0.147407,0.54,0.46,0.25,2.0,0.18,568,baseline,3
2,0.147407,0.54,0.46,0.25,2.0,0.15,556,baseline,3
3,0.147407,0.54,0.46,0.25,2.0,0.12,477,baseline,3
4,0.147407,0.54,0.46,0.25,2.0,0.22,590,baseline,3
5,0.147411,0.54,0.46,0.25,NaN,0.12,477,baseline,3
6,0.147411,0.54,0.46,0.25,NaN,0.15,556,baseline,3
7,0.147411,0.54,0.46,0.25,NaN,0.18,568,baseline,3
8,0.147411,0.54,0.46,0.25,NaN,0.20,572,baseline,3
9,0.147411,0.54,0.46,0.25,NaN,0.22,590,baseline,3


최적 최종 OOF MAE: 0.14740666666666666
기존 screen baseline MAE: 0.14725666666666665


## 9. 제출 후보 선택

제출 제한을 고려하여 완전히 비슷한 조합만 저장하지 않고 다음 성격을 나눕니다.

- `cv_best`: OOF 최적
- `safe`: 원본에 가까운 보수적 조합
- `feature_heavy`: 선택된 feature set + 낮은 Pair 비중
- `pair_heavy`: Pair 비중이 높은 후보
- `no_override`: override를 쓰지 않는 비교 후보


In [10]:
def row_to_config(
    row: pd.Series,
) -> dict:
    return {
        "tree_q": float(row["tree_q"]),
        "pair_q": float(row["pair_q"]),
        "pair_weight": float(row["pair_weight"]),
        "round_decimals": (
            None
            if pd.isna(row["round_decimals"])
            else int(row["round_decimals"])
        ),
        "override_threshold": (
            None
            if pd.isna(row["override_threshold"])
            else float(row["override_threshold"])
        ),
        "cv_mae": float(row["cv_mae"]),
        "feature_set": BEST_FEATURE_SET,
    }


candidate_configs = []

# 1. CV 최적
candidate_configs.append(
    ("cv_best", row_to_config(search_df.iloc[0]))
)

# 2. 원본에 가까운 safe
safe_rows = search_df[
    (search_df["tree_q"].between(0.52, 0.54))
    & (search_df["pair_q"] == 0.48)
    & (search_df["pair_weight"].between(0.15, 0.20))
    & (search_df["round_decimals"] == 2)
    & (
        search_df["override_threshold"]
        .isin([0.18, 0.20])
    )
]
if len(safe_rows):
    candidate_configs.append(
        ("safe", row_to_config(safe_rows.iloc[0]))
    )

# 3. ET/feature 중심
feature_rows = search_df[
    search_df["pair_weight"] == min(PAIR_WEIGHTS)
]
if len(feature_rows):
    candidate_configs.append(
        (
            "feature_heavy",
            row_to_config(feature_rows.iloc[0]),
        )
    )

# 4. Pair 중심
pair_rows = search_df[
    search_df["pair_weight"] == max(PAIR_WEIGHTS)
]
if len(pair_rows):
    candidate_configs.append(
        (
            "pair_heavy",
            row_to_config(pair_rows.iloc[0]),
        )
    )

# 5. override 없음
no_override_rows = search_df[
    search_df["override_threshold"].isna()
]
if len(no_override_rows):
    candidate_configs.append(
        (
            "no_override",
            row_to_config(no_override_rows.iloc[0]),
        )
    )

# 중복 설정 제거
unique_configs = []
seen = set()

for name, config in candidate_configs:
    key = (
        config["tree_q"],
        config["pair_q"],
        config["pair_weight"],
        config["round_decimals"],
        config["override_threshold"],
    )

    if key not in seen:
        seen.add(key)
        unique_configs.append((name, config))

FINAL_CONFIGS = unique_configs[:5]

print("최종 제출 후보")
for name, config in FINAL_CONFIGS:
    print(name, config)


최종 제출 후보
cv_best {'tree_q': 0.54, 'pair_q': 0.46, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14740666666666666, 'feature_set': 'baseline'}
safe {'tree_q': 0.54, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': 0.18, 'cv_mae': 0.14744333333333332, 'feature_set': 'baseline'}
feature_heavy {'tree_q': 0.5, 'pair_q': 0.5, 'pair_weight': 0.1, 'round_decimals': 2, 'override_threshold': 0.22, 'cv_mae': 0.14742333333333332, 'feature_set': 'baseline'}
no_override {'tree_q': 0.5, 'pair_q': 0.5, 'pair_weight': 0.1, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14754, 'feature_set': 'baseline'}


## 10. 전체 데이터 최종 학습 및 제출 파일 생성


In [11]:
# test/sample_submission 경로 재확인
if not sample_path.exists():
    print(
        "sample_submission.csv가 DATA_DIR에 없어서 "
        "train의 ID와 target 형식을 이용해 템플릿을 만듭니다."
    )
    submission_template = pd.DataFrame({
        ID_COLUMN: test_raw[ID_COLUMN],
        TARGET: 0.0,
    })
else:
    submission_template = pd.read_csv(sample_path)

full_bounds = compute_iqr_bounds(
    X_raw,
    WINSORIZE_COLUMNS,
)

train_full_w = apply_winsorize(
    X_raw,
    full_bounds,
)
test_full_w = apply_winsorize(
    X_test_raw,
    full_bounds,
)

train_final_features = build_feature_set(
    train_full_w,
    BEST_FEATURE_SET,
)
test_final_features = build_feature_set(
    test_full_w,
    BEST_FEATURE_SET,
)

test_tree_matrices = []

for seed in SEEDS:
    tree_matrix = fit_extra_trees_matrix(
        train_final_features,
        test_final_features,
        y,
        n_estimators=FINAL_N_ESTIMATORS,
        random_seed=seed,
    )
    test_tree_matrices.append(tree_matrix)
    print(f"전체 데이터 ExtraTrees seed {seed} 완료")

combined_test_tree_matrix = np.concatenate(
    test_tree_matrices,
    axis=1,
)

train_base = add_base_features(train_full_w)
test_base = add_base_features(test_full_w)

test_pair_matrix = pair_prediction_matrix(
    train_base,
    test_base,
    y,
)
print("전체 데이터 Pair 예측 완료")

test_nearest_dist, test_nearest_target = (
    nearest_neighbor_lookup(
        X_raw,
        X_test_raw,
        y,
    )
)
print("전체 데이터 nearest-neighbor lookup 완료")

saved_paths = []

for rank, (name, config) in enumerate(
    FINAL_CONFIGS,
    start=1,
):
    tree_pred = np.quantile(
        combined_test_tree_matrix,
        config["tree_q"],
        axis=1,
    )
    pair_pred = np.quantile(
        test_pair_matrix,
        config["pair_q"],
        axis=1,
    )

    blended = (
        (1.0 - config["pair_weight"]) * tree_pred
        + config["pair_weight"] * pair_pred
    )

    final_pred = apply_postprocess(
        blended,
        config["round_decimals"],
        config["override_threshold"],
        test_nearest_dist,
        test_nearest_target,
    )

    submission = submission_template.copy()
    submission[TARGET] = final_pred

    filename = (
        f"submit_feature_v6_rank{rank}_{name}"
        f"_fs-{BEST_FEATURE_SET}"
        f"_tq{config['tree_q']:.2f}"
        f"_pq{config['pair_q']:.2f}"
        f"_w{config['pair_weight']:.2f}"
        f"_th{config['override_threshold']}"
        ".csv"
    )

    output_file = OUTPUT_DIR / filename
    submission.to_csv(
        output_file,
        index=False,
        encoding="utf-8",
    )
    saved_paths.append(output_file)

    print("\nSaved:", output_file)
    print("Config:", config)
    print(
        "Prediction min/mean/max:",
        float(final_pred.min()),
        float(final_pred.mean()),
        float(final_pred.max()),
    )
    display(submission.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)


전체 데이터 ExtraTrees seed 42 완료
전체 데이터 ExtraTrees seed 52 완료
전체 데이터 ExtraTrees seed 62 완료
전체 데이터 Pair 예측 완료
전체 데이터 nearest-neighbor lookup 완료

Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank1_cv_best_fs-baseline_tq0.54_pq0.46_w0.25_th0.2.csv
Config: {'tree_q': 0.54, 'pair_q': 0.46, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14740666666666666, 'feature_set': 'baseline'}
Prediction min/mean/max: 0.0 0.5081966666666667 1.0


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.50
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank2_safe_fs-baseline_tq0.54_pq0.48_w0.20_th0.18.csv
Config: {'tree_q': 0.54, 'pair_q': 0.48, 'pair_weight': 0.2, 'round_decimals': 2, 'override_threshold': 0.18, 'cv_mae': 0.14744333333333332, 'feature_set': 'baseline'}
Prediction min/mean/max: 0.0 0.5120033333333334 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.50
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank3_feature_heavy_fs-baseline_tq0.50_pq0.50_w0.10_th0.22.csv
Config: {'tree_q': 0.5, 'pair_q': 0.5, 'pair_weight': 0.1, 'round_decimals': 2, 'override_threshold': 0.22, 'cv_mae': 0.14742333333333332, 'feature_set': 'baseline'}
Prediction min/mean/max: 0.0 0.4972266666666666 1.0


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank4_no_override_fs-baseline_tq0.50_pq0.50_w0.10_thNone.csv
Config: {'tree_q': 0.5, 'pair_q': 0.5, 'pair_weight': 0.1, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14754, 'feature_set': 'baseline'}
Prediction min/mean/max: 0.0 0.4972433333333333 1.0


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank1_cv_best_fs-baseline_tq0.54_pq0.46_w0.25_th0.2.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank2_safe_fs-baseline_tq0.54_pq0.48_w0.20_th0.18.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank3_feature_heavy_fs-baseline_tq0.50_pq0.50_w0.10_th0.22.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_feature_v6_rank4_no_override_fs-baseline_tq0.50_pq0.50_w0.10_thNone.csv


## 결과를 볼 때 확인할 부분

실행 후 다음 세 화면을 확인하면 됩니다.

1. `screen_df`
   - 어떤 feature set이 기준선보다 실제로 좋아졌는지
2. Feature importance 상위 40개
   - 새 파생변수가 실제로 사용됐는지
3. `search_df.head(30)`
   - 최종 seed ensemble의 OOF MAE와 최적 조합

새 파생변수가 기준선을 이기지 못하면 노트북은 자동으로 `baseline`을 선택합니다.  
즉 feature engineering이 효과가 없을 때 점수를 억지로 악화시키지 않도록 설계했습니다.
